# Load Dependencies

In [ ]:
import pandas as pd
import numpy as np
import os

# How to Load Data

## Creating dataframes from scratch

In [ ]:
# Create a dataframe from scratch
data = {
    'name': ['John', 'Anna', 'Peter', 'Linda'],
    'location': ['New York', 'Paris', 'Berlin', 'London'],
    'age': [24, 13, 53, 33]
}
df = pd.DataFrame(data)
df

## Loading CSV Files

In [ ]:
# Load a sample CSV file using the `read_csv` method. Excel files can similarly be loaded using `read_excel`.
df = pd.read_csv(os.path.join('..', '99_Resources', 'test_scores_example_data.csv'))
df

# Data Diagnostics

In [ ]:
# Display the number of rows and columns in the dataframe
df.shape

In [ ]:
# General information about the data, including the number of non-null entries and data types
df.info()

In [ ]:
# Compute basic statistics for numeric columns
df.describe()

In [ ]:
# List all unique values and their counts in a specific column
df['program'].value_counts(dropna=False) # The dropna=False argument includes NaN values in the count

In [ ]:
# Compute the median and mean of the 'test_score' column
print("Median and Mean of test_score:")
print(f"- Median:  {df['test_score'].median()}")
print(f"- Mean:    {df['test_score'].mean()}")

# Selecting and Filtering Data

## Selecting Columns

In [ ]:
# How do we slice and dice the data?
# We can select a specific column using the column name and the `[]` operator
df['test_score'] # Note that this outputs a Series

In [ ]:
# We can also select multiple columns by passing a list of column names
df[['student_name', 'test_score']] # Note that this outputs a DataFrame

## Selecting Rows
How about selecting rows? We can do this a few different ways.

### 1. Using `iloc`

In [ ]:
# `iloc` selects rows by absolute index number - remember, Python uses zero-based indexing!
df.iloc[0] # Select the first row
df.iloc[-1] # Select the last row
df.iloc[1:3] # Select the second and third rows

### 2. Using `loc`

In [ ]:
# `loc` selects rows by their actual unique label
# If the index is not set, the default index is just a range of integers starting from 0, 
# so the `loc` method behaves similarly to the `iloc` method
df.loc[0] # Select the first row
df.loc[1:3] # Select the second, third, AND fourth rows - this is inclusive of the last index!
# Note that using `-1` like with iloc doesn't make sense, since this is based on index 
# labels, not positions

### 3. Boolean Masks

In [ ]:
# Boolean indexing selects rows based on a logical condition
# For example, we can select all rows where the test score is greater than or equal to 
# 90 by creating a boolean mask with a logical operator
mask = df['test_score'] >= 90

# We can then use this mask to filter the dataframe. Note that this uses the same `[]` 
# operator as before, but now we are passing a boolean mask instead of a column name. 
# Don't worry, Python will know the difference!
df[mask] # Select all rows where the test score is greater than 90

# This is equivalent to the following:
df.loc[mask]
df.loc[df['test_score'] >= 90] # All at once is a great way to do it!

In [ ]:
# We can also use the `&` operator to combine multiple conditions
# For example, we can select all rows where the test score is greater than or equal to
# 90 AND the student age is less then 20
df[(df['test_score'] >= 90) & (df['age'] < 20)]

In [ ]:
# We can also use the `isin` method to select rows based on a list of values
# This is useful when you want to filter the dataframe based on multiple values in a column,
# such as selecting from categorical columns
df[df['program'].isin(['Engineering', 'Media'])]

### 4. Using `query`

In [ ]:
# The `query` method selects rows based on a condition expressed as a string
# This is a more readable way to filter the dataframe using a string expression, and 
# may be more intuitive for users familiar with SQL
df.query('test_score >= 90 and age < 20')

# Data Cleanup

In [ ]:
# We can drop rows with missing data
df.dropna(axis=0)

In [ ]:
# We can fill missing data with assumed values
df = df.fillna(value={'test_score': 50, 'age': df.age.median(), 'program': 'Undeclared'})
df

## Updating Data

In [ ]:
# We can add new columns to the dataframe using the `[]` operator
# For example, we can add a new column called 'pass' that indicates whether the student
# passed the test (score >= 60)
df['pass'] = df['test_score'] >= 60
df

In [ ]:
# We can also drop columns from the dataframe using the `drop` method
# For example, we can drop the 'pass' column we just created
df.drop(columns=['pass']) # This returns a new dataframe without the 'pass' column

In [ ]:
# We can identify duplicates in the dataframe using the `duplicated` method
# This is useful for identifying rows that are identical in all columns
# Note that this method returns a boolean Series indicating which rows are duplicates
df.duplicated() # This returns a boolean Series indicating which rows are duplicates
# Any duplicates returned? Use df.drop_duplicates(). This returns a new dataframe without the duplicate rows if they exist

# Data Grouping

In [ ]:
# Grouping data is a powerful feature of pandas and makes it easy to perform
# aggregations on the data
# For example, we can group the data by the 'program' column and compute the median
# test score for each program
grouped = df.groupby('program')['test_score'].median()
grouped

In [ ]:
# We can also group by multiple columns
grouped = df.groupby(['program', 'state'])['test_score'].median()
grouped

In [ ]:
# Groupbys can be turned into pivot tables by "unstacking" the data
df.groupby(['program', 'state']).size().unstack().fillna(0).astype(int)

# Merging Data

## Horizontal Joins

In [ ]:
# There are a variety of ways that we can merge dataframes together
# For example, we can use the `merge` method to combine two dataframes based on a common column
# or index, similar to SQL joins
# For the sake of example, let's merge the test dataframe with a copy of itself
df.merge(df, left_index=True, right_index=True, suffixes=('', '_right'))

## Vertical Joins

In [ ]:
# We can also use the `concat` method to concatenate multiple dataframes along a particular axis
# matching the index or columns
# For example, we can concatenate two dataframes vertically
df1 = pd.DataFrame({'A': [1, 2], 'B': [3, 4]})
df2 = pd.DataFrame({'A': [5, 6], 'B': [7, 8]})
df_concat = pd.concat([df1, df2], axis=0, ignore_index=True)
df_concat

# Exporting Data

## Basic Exports

In [ ]:
# Let's export to CSV
# Choose a filepath to export the data to
fp = 'sample_data.csv'
# Export the dataframe to a CSV file
df.to_csv(fp, index=False) # Do you want to include the index? (default is True)

## Advanced Exports

In [ ]:
# Let's export to Excel, using separate sheets for each unique program
# Choose a filepath to export the data to
fp = 'sample_data.xlsx'

# Identify all unique programs
unique_programs = df['program'].unique()

# We first need to create a Pandas Excel writer using XlsxWriter as the engine
# This will allow us to write multiple sheets to the same file
# The `with` statement ensures that the file is properly saved and closed
with pd.ExcelWriter('example_export.xlsx') as writer:
    # Loop through each unique program
    for program in unique_programs:
        # Filter the dataframe for the current program
        df_program = df[df['program'] == program]
        # Write the dataframe to a sheet named after the program
        df_program.to_excel(writer, sheet_name=program, index=False)